# Discrete-Adjoint PDE Control
### One gradient, independent of the number of controls

**Learning goal:** Formulate a bounded PDE-constrained optimization problem, derive and implement its discrete adjoint, verify the gradient using directional derivatives, and use L-BFGS-B to control a fixed-grid convection-diffusion state.

**Time:** 60-90 minutes. Python, NumPy, SciPy, Matplotlib. The spatial grid is fixed at N = 200; this is not a grid-convergence exercise.

**Core idea:** A direct sensitivity method would solve one linearized state equation for every design variable. The adjoint reorganizes the chain rule so the full scalar-objective gradient requires one transpose solve, regardless of the number of controls.

**Success targets:** adjoint and direct-sensitivity gradients agree to relative norm error below 1e-10; a centered directional-derivative check reaches relative error below 1e-6; the optimizer reports success and reduces the objective by at least 90%; and all bounds and algebraic residual checks pass.

Implementation cells intentionally remain TODOs. Derive the signs before coding; a plausible optimized plot does not validate an adjoint.

## 1. Fixed-grid state equation and control basis

Solve the steady one-dimensional convection-diffusion equation
$$-\nu u''(x)+a u'(x)=s(x;p),\qquad x\in(0,1),\qquad u(0)=u(1)=0,$$
with $\nu=0.01$, $a=1$, and N = 200 interior nodes. Use centered second diffusion and first-order upwind convection for positive a:
$$-\nu\frac{u_{i-1}-2u_i+u_{i+1}}{h^2}+a\frac{u_i-u_{i-1}}{h}=s_i.$$
This produces a nonsymmetric sparse system
$$Au=Bp,$$
where p contains M = 12 actuator amplitudes. Let column j of B be a Gaussian
$$B_{ij}=\exp\left[-\frac{(x_i-c_j)^2}{2\sigma^2}\right],$$
with equally spaced centers $c_j$ from 0.08 to 0.92 and $\sigma=0.045$. Use bounds $0\le p_j\le20$.

Use a target that cannot be represented exactly by the coarse actuator basis:
$$u_d(x)=0.8\exp[-((x-0.68)/0.10)^2]+0.22\exp[-((x-0.30)/0.06)^2].$$

Factor A and $A^T$ once. Do not form an inverse. Confirm A is nonsymmetric, has positive diagonal, and has at most 3N nonzeros.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags
from scipy.sparse.linalg import splu
from scipy.optimize import minimize

N = 200
M = 12
NU = 0.01
ADV = 1.0
SIGMA = 0.045
ALPHA = 2.0e-5
BETA = 5.0e-4
P_BOUNDS = (0.0, 20.0)

# TODO: Set a reproducible print/plot configuration and record SciPy version.


## 2. Objective and regularization

Minimize
$$J(u,p)=\frac{h}{2}\|u-u_d\|_2^2+\frac{\alpha}{2}\|p\|_2^2+\frac{\beta}{2}\|Dp\|_2^2,$$
where D is the first-difference matrix on controls: $(Dp)_j=p_{j+1}-p_j$. The first term tracks the desired state, alpha limits control effort, and beta discourages actuator-to-actuator roughness.

Implement a state solve and an objective decomposition returning tracking, effort, smoothness, and total terms. Use the same h weighting in the objective and adjoint. Omitting it changes the balance between tracking and regularization.

Start from $p_0=2\mathbf{1}$. Cache or reuse the factorizations, but do not cache states across different p values unless the key is exact and the implementation remains transparent.

In [ ]:
def build_problem(n=N, m=M):
    """Return x, h, sparse A, dense B, D, target, and LU factors."""
    # TODO: Assemble the prescribed fixed-grid problem and factor A/A.T.
    raise NotImplementedError

def solve_state(p, problem):
    # TODO: Solve A u = B p and return u.
    raise NotImplementedError

def objective_terms(u, p, problem):
    # TODO: Return tracking, effort, smoothness, total.
    raise NotImplementedError


## 3. Derive the discrete adjoint

Use the residual $R(u,p)=Bp-Au=0$ and Lagrangian
$$\mathcal L(u,p)=J(u,p)+\lambda^T(Bp-Au).$$
Stationarity with respect to u gives
$$A^T\lambda=h(u-u_d).$$
The reduced gradient is
$$\nabla_p\hat J=B^T\lambda+\alpha p+\beta D^TDp.$$

Derive these equations yourself and check the sign by differentiating $u=A^{-1}Bp$. Implement one routine that returns objective and gradient for SciPy. Count state and adjoint solves explicitly. A scalar objective requires one state and one adjoint solve per new p, not M state solves.

For an independent direct-sensitivity reference, solve $AS=B$ for all M columns and compute
$$\nabla_p\hat J=S^T h(u-u_d)+\alpha p+\beta D^TDp.$$
Use this only for verification at one p, not inside the optimizer.

In [ ]:
def objective_and_adjoint_gradient(p, problem, counters=None):
    """Return reduced objective and exact discrete-adjoint gradient."""
    # TODO: One state solve, one transpose solve, regularization, counters.
    raise NotImplementedError

def direct_sensitivity_gradient(p, problem):
    # TODO: Solve A S = B and form the independent reference gradient.
    raise NotImplementedError


## 4. Gradient verification

At a seeded interior test point $p_{test}\in[3,7]^M$, draw a seeded unit direction d. Compare:

1. The full adjoint gradient with the direct-sensitivity gradient.
2. The scalar adjoint projection $g^Td$ with centered finite differences
$$\phi_\epsilon=\frac{\hat J(p+\epsilon d)-\hat J(p-\epsilon d)}{2\epsilon}$$
for $\epsilon=10^{-1},10^{-2},\ldots,10^{-8}$.

Plot relative directional error
$$e(\epsilon)=\frac{|\phi_\epsilon-g^Td|}{\max(1,|g^Td|)}.$$
Expect a truncation-to-roundoff curve, not monotonic decrease to the smallest epsilon. Report the best epsilon and error. Keep the test point away from active bounds so this verifies the unconstrained reduced gradient.

Also verify the primal and adjoint relative residuals independently. Do not use a finite-difference gradient as the optimization gradient.

In [ ]:
# TODO: Perform direct-versus-adjoint and directional-derivative checks.
# Save the epsilon/error arrays for a log-log diagnostic plot.


## 5. Bounded optimization experiment

Run `scipy.optimize.minimize` with method `L-BFGS-B`, analytic `jac`, twelve bounds [0,20], `gtol=1e-8`, `ftol=1e-12`, and `maxiter=200`. Record objective components and projected-gradient information in a callback without recomputing a state solely for plotting.

Required diagnostics:

1. Initial state, target, and optimized state versus x.
2. Initial and optimized actuator amplitudes versus actuator center.
3. Objective, tracking, effort, and smoothness versus accepted iteration on a semilog axis.
4. Gradient-check error versus epsilon.

Report optimizer status/message, iterations, function/gradient calls, counted state/adjoint solves, objective reduction, active lower/upper bounds, and final state/adjoint residuals. Distinguish the raw gradient from the projected gradient when bounds are active.

In [ ]:
# TODO: Run L-BFGS-B, collect history without redundant PDE solves,
# print the summary table, and create the four required diagnostics.


## 6. Automatic success checks

Assert:

- Shapes are A(N,N), B(N,M), D(M-1,M); A and B entries are finite; A has at most 3N nonzeros and is nonsymmetric.
- Every state and adjoint relative residual is below 1e-11.
- Adjoint and direct-sensitivity gradients agree to relative norm error below 1e-10 at the seeded interior test point.
- The best centered directional-derivative error is below 1e-6, and its epsilon is reported rather than assumed.
- The optimizer reports success; all p values satisfy bounds within 1e-10; the final objective is below 10% of the initial objective.
- Recomputed final objective components sum to the reported total within 1e-12 relative error.
- The analytic gradient uses exactly one adjoint solve per new objective/gradient evaluation; no loop over controls appears in the adjoint routine.

Do not require a particular active-set pattern or iteration count. If the 90% reduction check fails, diagnose signs, h weighting, target construction, and optimizer bookkeeping before changing the problem.

In [ ]:
# TODO: Implement assertions tied to the actual model and optimization result.


## 7. Interpretation, stretch goal, and references

Answer briefly:

1. Why does the adjoint cost not grow linearly with the number of controls for one scalar objective?
2. Why is the transpose system essential for this upwind, nonsymmetric operator?
3. What does beta change in the control field and tracking error?
4. How do active bounds alter the correct optimality condition?
5. Which pieces generalize directly to a nonlinear CFD residual $R(U,p)=0$, and which require linearization at every design iterate?

**Stretch goal (15-25 minutes):** Repeat with M = 48 actuators while keeping N fixed. Compare direct-sensitivity and adjoint solve counts and wall times for one gradient. Do not change the PDE grid. Alternatively, add one scalar equality constraint on total control and solve with SLSQP using its analytic gradient.

### References

- **Dedicated project reference:** Michael B. Giles and Niles A. Pierce, "An Introduction to the Adjoint Approach to Design," *Flow, Turbulence and Combustion* 65, 393-415, 2000. DOI: 10.1023/A:1011430410075. [Author PDF](https://people.maths.ox.ac.uk/gilesm/files/ftc00.pdf). Focus on the Lagrangian derivation, cost scaling, and discrete-versus-continuous distinction.
- **Optimizer reference:** [SciPy L-BFGS-B documentation](https://docs.scipy.org/doc/scipy/reference/optimize.minimize-lbfgsb.html). Record version, stopping tolerances, bounds, and final status.
- **Paper to read:** Christian Psenica, Lean Fang, Seth Zoppelt, Mark Leader, and Ping He, "A Modular Conjugate Heat Transfer Optimization Framework for Thermal Management of Electric Aircraft," *International Journal of Heat and Mass Transfer* 259, 128345, 2026. DOI: 10.1016/j.ijheatmasstransfer.2026.128345. [NASA Glenn accepted manuscript and metadata](https://ntrs.nasa.gov/citations/20260000381). Read for modular multiphysics coupling, discrete-adjoint gradients, interface consistency, constraints, and fine/unsteady confirmation of optimized designs.

### Topic ledger

| Date / record | Primary objective | Status |
|---|---|---|
| Older assignments, dates unavailable | Grid convergence and errors | User reported five repetitive assignments; individual records unavailable |
| 2026-10-05 | Grid convergence and discretization uncertainty | Inspected |
| 2026-10-06 corrected | Fixed-mesh advection, CFL, conservation, positivity, TV | Inspected |
| 2026-10-07 | Krylov methods and line-block preconditioning | Inspected |
| 2026-10-08 | Data layout, cache behavior, and kernel throughput | Inspected |
| 2026-10-09 | Nonlinear natural-circulation thermal-fluid modeling | Inspected |
| 2026-10-12 | Discrete-adjoint PDE-constrained optimization | This assignment |

Only five preceding distinct projects are recoverable, so a complete seven-project audit remains unavailable. The reported older repetitions remain excluded. This is assignment 4 of the ten-assignment exclusion established October 6; six remain after today. Planned next primary objectives: multiphysics coupling, high-order reconstruction, parallel algorithms, then time integration. No grid-refinement, observed-order, Richardson, or GCI task appears here.